# Lap-Time Performance Decomposition — narrative run-through

This notebook is **thin on purpose**: every bit of logic lives in `src/`, and
this file only imports it, runs the pipeline, and narrates the result. That keeps
the analysis testable and re-runnable from the command line (`python -m src.run`).

> Live FastF1 data needs network access once (then it's cached). Set
> `USE_SYNTHETIC = True` to run the whole pipeline offline on a fixture with a
> known ground truth.

In [ ]:
import sys; sys.path.append('..')  # so `import config` / `from src import ...` resolve
import config
from src import run
from IPython.display import Image, display

USE_SYNTHETIC = True   # flip to False for live FastF1 data
f"{config.DRIVER_A} vs {config.DRIVER_B} — {config.YEAR} {config.GRAND_PRIX} {config.SESSION}"

## 1. Run the pipeline

`run_pipeline` loads clean laps, resamples both drivers onto a common distance
grid, builds the cumulative time-delta curve, **reconciles its endpoint against
the official lap-time gap** (it raises if that fails), decomposes the lap into
micro-sectors, bootstraps a CI per sector, and attributes the key ones.

In [ ]:
res = run.run_pipeline(use_synthetic=USE_SYNTHETIC)
print(f"official gap : {res['official_gap']:+.3f} s")
print(f"curve endpoint: {res['endpoint']:+.3f} s  (residual {res['residual']:+.4f} s)")
print(f"clean laps    : {config.DRIVER_A}={res['n_laps_a']}, {config.DRIVER_B}={res['n_laps_b']}")

## 2. Where the gap comes from — and is it real?

Each micro-sector's mean delta with its 95% bootstrap CI. `significant=True`
means the interval excludes zero — a repeatable advantage, not lap-to-lap noise.

In [ ]:
res['table'][['rank','sector','mid_m','delta_s_mean','ci_low','ci_high','significant']].head(10)

## 3. Figures

In [ ]:
run.write_outputs(res)
for fig in ['cumulative_delta','sector_deltas','input_overlays','track_map']:
    p = config.FIGURES_DIR / f'{fig}.png'
    if p.exists(): display(Image(filename=str(p)))

## 4. Driver-input attribution at the key micro-sectors

For the largest *real* deltas, what differs in the inputs and why.

In [ ]:
for n in res['attrib'].get('narrative', []):
    print('-', n)

## 5. Findings injected into the report

The same numbers are written into `REPORT.md` between the AUTOGEN markers, so the
narrative report stays in sync with the data on every run.

In [ ]:
run.inject_report(res)
print('REPORT.md findings section updated.')